In [1]:
!pip install transformers datasets -q


In [2]:
# =========================
# INSTALL (run once)
# =========================
!pip install transformers datasets -q


# =========================
# IMPORTS
# =========================
import torch
from transformers import BertTokenizerFast, BertForQuestionAnswering
from transformers import Trainer, TrainingArguments
from datasets import load_dataset


# =========================
# DEVICE SETUP (IMPORTANT)
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =========================
# LOAD DATASET
# =========================
dataset = load_dataset("squad")

# Reduce size (Colab friendly)
train_dataset = dataset["train"].select(range(2000))
val_dataset = dataset["validation"].select(range(1000))


# =========================
# TOKENIZER
# =========================
tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")


# =========================
# PREPROCESS FUNCTION
# =========================
def preprocess(example):
    inputs = tokenizer(
        example["question"],
        example["context"],
        truncation=True,
        padding="max_length",
        max_length=384,
        return_offsets_mapping=True
    )

    offsets = inputs.pop("offset_mapping")
    start_positions = []
    end_positions = []

    for i, offset in enumerate(offsets):
        answer = example["answers"][i]
        start_char = answer["answer_start"][0]
        end_char = start_char + len(answer["text"][0])

        sequence_ids = inputs.sequence_ids(i)

        # Find context start and end
        idx = 0
        while sequence_ids[idx] != 1:
            idx += 1
        context_start = idx

        while sequence_ids[idx] == 1:
            idx += 1
        context_end = idx - 1

        # If answer not found
        if offset[context_start][0] > end_char or offset[context_end][1] < start_char:
            start_positions.append(0)
            end_positions.append(0)
        else:
            for j in range(context_start, context_end + 1):
                if offset[j][0] <= start_char < offset[j][1]:
                    start_positions.append(j)
                if offset[j][0] < end_char <= offset[j][1]:
                    end_positions.append(j)

    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions

    return inputs


# Apply preprocessing
train_dataset = train_dataset.map(preprocess, batched=True, remove_columns=train_dataset.column_names)
val_dataset = val_dataset.map(preprocess, batched=True, remove_columns=val_dataset.column_names)


# =========================
# FORMAT FOR PYTORCH
# =========================
train_dataset.set_format(type="torch")
val_dataset.set_format(type="torch")


# =========================
# MODEL
# =========================
model = BertForQuestionAnswering.from_pretrained("bert-base-uncased")
model.to(device)   # 🔥 move model to GPU


# =========================
# TRAINING ARGUMENTS
# =========================
training_args = TrainingArguments(
    output_dir="./qa_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    logging_dir="./logs",
)


# =========================
# TRAINER
# =========================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)


# =========================
# TRAIN
# =========================
trainer.train()


# =========================
# TEST (INFERENCE)
# =========================
question = "Where do I live?"
context = "My name is Yash and I live in Delhi."

inputs = tokenizer(question, context, return_tensors="pt")

# 🔥 Move inputs to SAME device
inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model(**inputs)

start = torch.argmax(outputs.start_logits).item()
end = torch.argmax(outputs.end_logits).item()

answer = tokenizer.convert_tokens_to_string(
    tokenizer.convert_ids_to_tokens(inputs["input_ids"][0][start:end+1])
)

print("Answer:", answer)

Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/1.82M [00:00<?, ?B/s]

...

BertForQuestionAnswering LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
qa_outputs.weight                          | MISSING    | 
qa_outputs.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized beca

Epoch,Training Loss,Validation Loss
1,No log,1.735767
2,1.931323,1.677844


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Answer: delhi
